[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_prefill_decode_separation.ipynb)

# 🔴 困难: Prefill-Decode 分离推理

实现 **Prefill-Decode分离** 的推理过程——将LLM推理分为预填充 (Prefill) 和解码 (Decode) 两个阶段，这是大模型高性能部署的核心技术。

**Prefill阶段**：一次性处理输入 prompt 的所有 token，生成第一个输出 token 和 KV Cache。

**Decode阶段**：自回归生成后续 token，每步只处理一个新 token，复用 KV Cache。

### 关键概念
- **KV Cache**: 存储每层的Key和Value张量，形状为 `(batch_size, num_kv_heads, seq_len, head_dim)`
- **因果掩码**: 确保 token 只能关注之前的 token，Prefill 使用下三角矩阵，Decode 使用单元素掩码
- **位置编码**: 每个token需要正确的位置ID，Prefill阶段为 `[0, 1, ..., seq_len-1]`

### 函数签名
```python
def prefill_decode_inference(model, tokenizer, prompt: str, max_new_tokens: int = 128):
    # model: Qwen3ForCausalLM实例
    # tokenizer: AutoTokenizer实例
    # prompt: 输入文本
    # max_new_tokens: 最大生成token数
    # 返回: 生成的完整文本
```

### 要求
1. 正确实现Prefill和Decode两个阶段
2. 正确管理和更新KV Cache
3. 使用因果掩码保证自回归特性，必须实现
4. 处理EOS token提前终止

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge transformers')
except ImportError:
    pass

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, Qwen3ForCausalLM
from transformers.models.qwen3 import Qwen3ForCausalLM  # 注意：这需要正确的导入

In [ ]:
# 辅助函数：创建因果掩码
def create_causal_mask(seq_len, device):
    """创建下三角因果掩码"""
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask

def create_single_token_mask(seq_len, pos, device):
    """为decode阶段创建单token掩码"""
    mask = torch.zeros(1, seq_len, device=device)
    mask[0, pos] = 1.0
    return mask

In [ ]:
# ✏️ 在此实现你的代码

def prefill_decode_inference(model, tokenizer, prompt: str, max_new_tokens: int = 128):
    # 1. 应用chat template
    # 2. Tokenize输入
    # 3. Prefill阶段：处理所有输入token
    # 4. 获取第一个生成token
    # 5. Decode阶段：自回归生成
    # 6. 返回完整生成的文本
    
    model.eval()
    device = next(model.parameters()).device
    
    # 在这里实现你的代码

In [ ]:
def prefill_decode_inference(model, tokenizer, prompt: str, max_new_tokens: int = 128):
    """
    执行Prefill-Decode分离推理

    Args:
        model: 模型
        tokenizer: 分词器
        prompt: 输入提示
        max_new_tokens: 最大生成长度
        
    Returns:
        生成的文本
    """
    model.eval()
    input_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,  # 应该设为True
    )
    print(input_text)

    num_head = model.config.num_attention_heads 
    head_dim = getattr(model.config, "head_dim", model.config.hidden_size // num_head)
    num_key_value_heads = model.config.num_key_value_heads
    num_layers = model.config.num_hidden_layers

    with torch.inference_mode():
        # Prefill
        input_ids = tokenizer(input_text, return_tensors="pt")["input_ids"].cuda()
        input_seq_len = input_ids.shape[1]
        
        # 修正1: position_ids 应该从0开始
        position_ids = torch.arange(0, input_seq_len, dtype=torch.long, device=input_ids.device).unsqueeze(0)
        
        # 计算旋转位置编码
        cos_emb, sin_emb = model.model.rotary_emb(
            torch.empty((1,), dtype=torch.bfloat16, device=input_ids.device), 
            position_ids
        )
        
        # 修正2: attention_mask 应该是因果掩码，形状为 (1, 1, seq_len, seq_len)
        attention_mask = torch.tril(torch.ones((1, 1, input_seq_len, input_seq_len), device=input_ids.device))
        
        # 修正3: KV cache 形状改为 (batch_size, num_kv_heads, seq_len, head_dim)
        kvcache_shape = (1, num_key_value_heads, input_seq_len, head_dim)
        past_key_values = [
            torch.zeros(kvcache_shape, dtype=torch.bfloat16, device=input_ids.device) 
            for _ in range(num_layers)
        ] + [
            torch.zeros(kvcache_shape, dtype=torch.bfloat16, device=input_ids.device) 
            for _ in range(num_layers)
        ]

        logits, past_key_values = model(
            input_ids=input_ids, 
            cos_emb=cos_emb,
            sin_emb=sin_emb,
            attention_mask=attention_mask,
            past_key_values=past_key_values
        )
        next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
        print(tokenizer.decode(next_token_id), end='', flush=True)
        
        # Decode
        generated_tokens = [next_token_id.item()]
        for step in range(128):
            if next_token_id.item() == tokenizer.eos_token_id:
                break

            # 修正4: position_ids 应该是当前token的位置
            current_pos = input_seq_len + step
            position_ids = torch.tensor([[current_pos]], dtype=torch.long, device=input_ids.device)
            
            cos_emb, sin_emb = model.model.rotary_emb(
                torch.empty((1,), dtype=torch.bfloat16, device=input_ids.device), 
                position_ids
            )
            
            # 修正5: attention_mask 需要扩展到新的序列长度
            total_seq_len = input_seq_len + step + 1
            attention_mask = torch.tril(torch.ones((1, 1, total_seq_len, total_seq_len), device=input_ids.device))
            
            logits, past_key_values = model(
                input_ids=next_token_id.unsqueeze(1), 
                cos_emb=cos_emb,
                sin_emb=sin_emb,
                attention_mask=attention_mask,
                past_key_values=past_key_values
            )
            next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
            print(tokenizer.decode(next_token_id), end='', flush=True)
            generated_tokens.append(next_token_id.item())
    full_text = tokenizer.decode(generated_tokens)
    return full_text

In [ ]:
# 🧪 调试 - 加载模型并测试

model_path = "/hy-tmp/hz/models-hub/Qwen3-0.6B"  # 或者使用HuggingFace路径
try:
    model = Qwen3ForCausalLM.from_pretrained(model_path).eval().cuda()
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    prompt = "Who is Donald Trump?"
    generated_text = prefill_decode_inference(model, tokenizer, prompt, max_new_tokens=50)
    print("Generated text:", generated_text)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure the model path is correct or use a different model")

### PD 分离

```bash
PD分离（Prefill-Decode Disaggregation）
├── 逻辑分离（您代码中的实现）
│   ├── Prefill阶段：处理完整prompt
│   ├── Decode阶段：逐个生成token
│   └── 共享KV Cache
├── 物理分离（高级优化）
│   ├── Prefill节点：专门处理预填充
│   ├── Decode节点：专门处理解码
│   └── KV Cache传输：Prefill→Decode传递缓存
└── 混合分离
    ├── 分块预填充（Chunked Prefill）
    ├── 持续批处理（Continuous Batching）
    └── 动态调度
```

In [ ]:
import numpy as np

def hf_inference(model_path: str, prompt: str):
    model = Qwen3ForCausalLM.from_pretrained(model_path).eval().cuda()
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    input_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,  # 应该设为True
    )
    print(input_text)

    num_head = model.config.num_attention_heads 
    head_dim = getattr(model.config, "head_dim", model.config.hidden_size // num_head)
    num_key_value_heads = model.config.num_key_value_heads
    num_layers = model.config.num_hidden_layers

    with torch.inference_mode():
        # Prefill
        input_ids = tokenizer(input_text, return_tensors="pt")["input_ids"].cuda()
        input_seq_len = input_ids.shape[1]
        
        # 修正1: position_ids 应该从0开始
        position_ids = torch.arange(0, input_seq_len, dtype=torch.long, device=input_ids.device).unsqueeze(0)
        
        # 计算旋转位置编码
        cos_emb, sin_emb = model.model.rotary_emb(
            torch.empty((1,), dtype=torch.bfloat16, device=input_ids.device), 
            position_ids
        )
        
        # 修正2: attention_mask 应该是因果掩码，形状为 (1, 1, seq_len, seq_len)
        attention_mask = torch.tril(torch.ones((1, 1, input_seq_len, input_seq_len), device=input_ids.device))
        
        # 修正3: KV cache 形状改为 (batch_size, num_kv_heads, seq_len, head_dim)
        kvcache_shape = (1, num_key_value_heads, input_seq_len, head_dim)
        past_key_values = [
            torch.zeros(kvcache_shape, dtype=torch.bfloat16, device=input_ids.device) 
            for _ in range(num_layers)
        ] + [
            torch.zeros(kvcache_shape, dtype=torch.bfloat16, device=input_ids.device) 
            for _ in range(num_layers)
        ]

        logits, past_key_values = model(
            input_ids=input_ids, 
            cos_emb=cos_emb,
            sin_emb=sin_emb,
            attention_mask=attention_mask,
            past_key_values=past_key_values
        )
        next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
        print(tokenizer.decode(next_token_id), end='', flush=True)
        
        # Decode
        generated_tokens = [next_token_id.item()]
        for step in range(128):
            if next_token_id.item() == tokenizer.eos_token_id:
                break

            # 修正4: position_ids 应该是当前token的位置
            current_pos = input_seq_len + step
            position_ids = torch.tensor([[current_pos]], dtype=torch.long, device=input_ids.device)
            
            cos_emb, sin_emb = model.model.rotary_emb(
                torch.empty((1,), dtype=torch.bfloat16, device=input_ids.device), 
                position_ids
            )
            
            # 修正5: attention_mask 需要扩展到新的序列长度
            total_seq_len = input_seq_len + step + 1
            attention_mask = torch.tril(torch.ones((1, 1, total_seq_len, total_seq_len), device=input_ids.device))
            
            logits, past_key_values = model(
                input_ids=next_token_id.unsqueeze(1), 
                cos_emb=cos_emb,
                sin_emb=sin_emb,
                attention_mask=attention_mask,
                past_key_values=past_key_values
            )
            next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
            print(tokenizer.decode(next_token_id), end='', flush=True)
            generated_tokens.append(next_token_id.item())
        
        print()  # 换行


def onnx_inference(model_path: str, prefill_model: str, decode_model: str, prompt: str):
    import onnxruntime
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    model = Qwen3ForCausalLM.from_pretrained(model_path).eval()
    prefill_session = onnxruntime.InferenceSession(prefill_model, providers=["CUDAExecutionProvider"])
    decode_session = onnxruntime.InferenceSession(decode_model, providers=["CUDAExecutionProvider"])

    input_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    print(input_text)

    num_head = model.config.num_attention_heads 
    head_dim = getattr(model.config, "head_dim", model.config.hidden_size // num_head)
    num_key_value_heads = model.config.num_key_value_heads
    num_layers = model.config.num_hidden_layers

    with torch.inference_mode():
        # Prefill
        input_ids = tokenizer(input_text, return_tensors="pt")["input_ids"]
        input_seq_len = input_ids.shape[1]
        
        position_ids = torch.arange(0, input_seq_len, dtype=torch.long).unsqueeze(0)
        cos_emb, sin_emb = model.model.rotary_emb(
            torch.empty((1,), dtype=torch.bfloat16), 
            position_ids
        )
        attention_mask = torch.tril(torch.ones((1, 1, input_seq_len, input_seq_len)))

        # 修正6: KV cache 形状使用当前序列长度
        kvcache_shape = (1, num_key_value_heads, input_seq_len, head_dim)
        past_key_values = [
            torch.zeros(kvcache_shape, dtype=torch.bfloat16) 
            for _ in range(num_layers)
        ] + [
            torch.zeros(kvcache_shape, dtype=torch.bfloat16) 
            for _ in range(num_layers)
        ]

        # Prefill推理
        prefill_inputs = {
            "input_ids": input_ids.cpu().numpy().astype(np.int64),
            "cos_emb": cos_emb.cpu().numpy().astype(np.float32),
            "sin_emb": sin_emb.cpu().numpy().astype(np.float32),
            "attention_mask": attention_mask.cpu().numpy().astype(np.float32),
        }
        for i in range(num_layers):
            prefill_inputs[f"past_key_cache_{i}"] = past_key_values[i].cpu().numpy().astype(np.float32)
            prefill_inputs[f"past_value_cache_{i}"] = past_key_values[i + num_layers].cpu().numpy().astype(np.float32)
        
        prefill_outputs = prefill_session.run(None, prefill_inputs)
        
        # 获取第一个token
        logits = torch.from_numpy(prefill_outputs[0])
        next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
        print(tokenizer.decode(next_token_id), end='', flush=True)
        
        # 更新KV cache
        updated_kv_cache = []
        for i in range(num_layers):
            updated_kv_cache.append(torch.from_numpy(prefill_outputs[1 + i * 2]))
        for i in range(num_layers):
            updated_kv_cache.append(torch.from_numpy(prefill_outputs[1 + num_layers + i * 2]))
        
        # Decode
        total_seq_len = input_seq_len
        for step in range(128):
            if next_token_id.item() == tokenizer.eos_token_id:
                break

            total_seq_len += 1
            current_pos = total_seq_len - 1
            
            # 修正7: 解码时只计算当前token的位置编码
            position_ids = torch.tensor([[current_pos]], dtype=torch.long)
            cos_emb, sin_emb = model.model.rotary_emb(
                torch.empty((1,), dtype=torch.bfloat16), 
                position_ids
            )
            
            # 修正8: attention_mask 需要扩展到新的序列长度
            attention_mask = torch.tril(torch.ones((1, 1, total_seq_len, total_seq_len)))
            
            # 修正9: 准备decode输入，使用更新后的KV cache
            decode_inputs = {
                "input_ids": next_token_id.unsqueeze(0).cpu().numpy().astype(np.int64),
                "cos_emb": cos_emb.cpu().numpy().astype(np.float32),
                "sin_emb": sin_emb.cpu().numpy().astype(np.float32),
                "attention_mask": attention_mask.cpu().numpy().astype(np.float32),
            }
            for i in range(num_layers):
                # 修正10: 确保KV cache形状匹配当前序列长度
                current_kv = updated_kv_cache[i]
                decode_inputs[f"past_key_cache_{i}"] = current_kv.cpu().numpy().astype(np.float32)
                decode_inputs[f"past_value_cache_{i}"] = updated_kv_cache[i + num_layers].cpu().numpy().astype(np.float32)
            
            decode_outputs = decode_session.run(None, decode_inputs)
            
            # 更新token
            logits = torch.from_numpy(decode_outputs[0])
            next_token_id = torch.argmax(logits[:, -1, :], dim=-1)
            print(tokenizer.decode(next_token_id), end='', flush=True)
            
            # 更新KV cache
            updated_kv_cache = []
            for i in range(num_layers):
                updated_kv_cache.append(torch.from_numpy(decode_outputs[1 + i * 2]))
            for i in range(num_layers):
                updated_kv_cache.append(torch.from_numpy(decode_outputs[1 + num_layers + i * 2]))
        
        print()  # 换行


if __name__ == '__main__':
    path = "/root/Qwen3-4B-Instruct-2507"
    prompt = "Who is Donald Trump?"
    
    # 测试HF推理
    print("=== HF Inference ===")
    hf_inference(path, prompt)
    
    # 测试ONNX推理（需要先导出ONNX模型）
    # decode_model = "qwen3_model_decoder.onnx"
    # prefill_model = "qwen3_model_prefill.onnx"
    # print("=== ONNX Inference ===")
    # onnx_inference(path, prefill_model, decode_model, prompt)

In [ ]:
# ✅ 提交 - 验证实现
from torch_judge import check
check('prefill_decode_inference')